# Final AFT goodness-of-fit bootstrap and Cox–Snell diagnostics

This independent notebook performs the final validation stage for the
41 pair–distribution models produced by the sequential AFT-selection
notebook.

It:

1. reads `data3.xlsx` and the 41-model sequential-AFT workbook;
2. refits the exact final covariate set recorded for every model and
   audits the refitted AIC against the prior workbook;
3. ranks the final AFT models **within each pair**;
4. retains the minimum-AIC model and every competitor with
   \(\Delta\mathrm{AIC}<2\);
5. performs parametric-bootstrap KS and Anderson–Darling tests on the
   conditional probability-integral-transform values, holding the
   observed covariates fixed and refitting the complete AFT model in
   every replicate;
6. calculates Cox–Snell residuals and plots their Nelson–Aalen
   cumulative hazard against the 45-degree reference line;
7. refits and bootstraps the corresponding pooled `All`, `PR`, and
   `BTW` distributions, using the same candidate-family set and
   bootstrap procedure; and
8. exports manuscript-ready tables and a multipage diagnostic PDF.

The AFT convention is

\[
T_i=\exp(z_i^\top\beta)T_{0i},\qquad
F(t_i\mid z_i)=F_0\!\left(t_i e^{-z_i^\top\beta}\right).
\]

The final bootstrap therefore tests

\[
u_i=F_0\!\left(t_i e^{-z_i^\top\beta}\right)
\sim \operatorname{Uniform}(0,1).
\]

Cox–Snell residuals are

\[
r_i=-\log S_0\!\left(t_i e^{-z_i^\top\beta}\right),
\]

which should approximately follow a unit-exponential distribution for
a correctly specified model. Because the present data contain no
censoring indicator, every residual is treated as an observed event.

> **Interpretation caution:** AIC values are compared only among models
> fitted to the same pair. Pooled and pair-specific AIC values are not
> directly compared because their sample scopes differ. Bootstrap
> p-values are adequacy tests, not effect-size or fit-strength measures.


In [1]:
# ============================================================
# 1. Imports, settings, input discovery, and data validation
# ============================================================

from pathlib import Path
import hashlib
import math
import os
import re
import textwrap
import time
import warnings

import numpy as np
import pandas as pd
from scipy import optimize, stats

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

from openpyxl import load_workbook
from openpyxl.formatting.rule import CellIsRule
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

warnings.filterwarnings("ignore", category=RuntimeWarning)


# -------------------- user-adjustable settings --------------------

ALPHA_GOF = 0.05

# The user's requested evidence set is the best final AFT model plus
# every competitor with a final within-pair ΔAIC strictly below 2.
DELTA_AIC_CUTOFF = 2.0
CUTOFF_IS_INCLUSIVE = False

# Final default. Override without editing the notebook, for example:
#   Windows: set HEADWAY_FINAL_BOOT=999
#   macOS/Linux: export HEADWAY_FINAL_BOOT=999
N_BOOT_FINAL = int(os.environ.get("HEADWAY_FINAL_BOOT", "499"))
N_BOOT_POOLED = int(
    os.environ.get("HEADWAY_POOLED_BOOT", str(N_BOOT_FINAL))
)
MIN_VALID_BOOT_FRACTION = 0.80

RANDOM_SEED = 20260731
SITE_ONE_LEVEL = "Tikatuli"
AIC_AUDIT_TOLERANCE = 1e-3
POSITIVE_SUPPORT_TOLERANCE = 1e-6

# Cox–Snell visualization settings.
COX_SNELL_ENVELOPE_SIMULATIONS = int(
    os.environ.get("HEADWAY_COX_ENVELOPE", "1000")
)
COX_SNELL_PLOT_QUANTILE = 0.99
COX_SNELL_MAX_X = 10.0

# Testing hooks. Leave at 0 for the complete analysis.
FINAL_MODEL_LIMIT = int(
    os.environ.get("HEADWAY_FINAL_MODEL_LIMIT", "0")
)
POOLED_MODEL_LIMIT = int(
    os.environ.get("HEADWAY_POOLED_MODEL_LIMIT", "0")
)

# Explicit paths are optional. When left as None, the notebook detects
# the correct workbooks from the current folder and common subfolders.
AFT_WORKBOOK = r"D:\Headway/final run\tables\T5_41_Sequential_AFT_Selection_VIF.xlsx"
DATA_WORKBOOK = r"D:\Headway\data3.xlsx"
POOLED_REFERENCE_WORKBOOK = r"D:\Headway\Tables\T1_Baseline_Headway_Distributions.xlsx"

OUTPUT_ROOT = Path(
    os.environ.get("HEADWAY_FINAL_OUTPUT_DIR", ".")
)
TABLES_DIR = OUTPUT_ROOT / "Tables"
FIGURES_DIR = OUTPUT_ROOT / "Figures"
INDIVIDUAL_FIGURES_DIR = FIGURES_DIR / "T6_CoxSnell_Selected_AFT"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
INDIVIDUAL_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_XLSX = (
    TABLES_DIR
    / "T6_Final_AFT_Bootstrap_CoxSnell.xlsx"
)
OUTPUT_PDF = (
    FIGURES_DIR
    / "T6_CoxSnell_Selected_AFT.pdf"
)


HEADWAY = "Time_Headway"
REQUIRED_DATA_COLUMNS = {
    "Pair",
    HEADWAY,
    "V_Target",
    "Target_Speed_km/hr",
    "Leading_Speed_km/hr",
    "Speed_Difference",
    "Occupancy",
    "Off_centeredness",
    "Site",
    "Flow_pcu/hr/m",
}

COVARIATES = [
    "Target_Speed_km/hr",
    "Leading_Speed_km/hr",
    "Speed_Difference",
    "Occupancy",
    "Off_centeredness",
    "Site",
    "Flow_pcu/hr/m",
]

CONTINUOUS_COVARIATES = {
    "Target_Speed_km/hr",
    "Leading_Speed_km/hr",
    "Speed_Difference",
    "Flow_pcu/hr/m",
}

COVARIATE_LABELS = {
    "Target_Speed_km/hr": "Target Vehicle Speed",
    "Leading_Speed_km/hr": "Leading Vehicle Speed",
    "Speed_Difference": "Speed Difference",
    "Occupancy": "Occupancy",
    "Off_centeredness": "Off-centeredness",
    "Site": "Site",
    "Flow_pcu/hr/m": "Flow",
}


def stable_seed(*parts):
    token = "|".join(map(str, parts)).encode("utf-8")
    offset = int(hashlib.sha256(token).hexdigest()[:8], 16)
    return int((RANDOM_SEED + offset) % (2**32 - 1))


def workbook_candidates():
    roots = [
        Path.cwd(),
        Path.cwd() / "Tables",
        Path.cwd() / "upload",
        Path.cwd() / "project_sources",
    ]
    paths = []
    seen = set()
    for root in roots:
        if not root.exists():
            continue
        for path in root.glob("*.xlsx"):
            if path.name.startswith("~$"):
                continue
            resolved = path.resolve()
            if resolved not in seen:
                seen.add(resolved)
                paths.append(resolved)
    return paths


def is_aft_workbook(path):
    try:
        excel_file = pd.ExcelFile(path)
        if "S1_Model_summary" not in excel_file.sheet_names:
            return False
        frame = pd.read_excel(
            path,
            sheet_name="S1_Model_summary",
            nrows=60,
        )
        required = {
            "Pair",
            "Distribution",
            "Distribution key",
            "Final covariates",
            "Final AIC",
            "Final k",
            "Final logLik",
        }
        return (
            required.issubset(frame.columns)
            and len(frame) == 41
        )
    except Exception:
        return False


def is_data_workbook(path):
    try:
        frame = pd.read_excel(path, nrows=5)
        return REQUIRED_DATA_COLUMNS.issubset(frame.columns)
    except Exception:
        return False


def is_pooled_reference_workbook(path):
    try:
        excel_file = pd.ExcelFile(path)
        return {
            "S3_Fits_all",
            "S6_MS_overall",
        }.issubset(excel_file.sheet_names)
    except Exception:
        return False


def resolve_input(explicit_path, predicate, preferred_names):
    if explicit_path:
        path = Path(explicit_path).expanduser().resolve()
        if not path.exists():
            raise FileNotFoundError(path)
        if not predicate(path):
            raise ValueError(
                f"Workbook does not have the required schema: {path}"
            )
        return path

    candidates = workbook_candidates()
    for preferred_name in preferred_names:
        for path in candidates:
            if (
                path.name.lower() == preferred_name.lower()
                and predicate(path)
            ):
                return path

    matches = [path for path in candidates if predicate(path)]
    if not matches:
        return None
    return max(matches, key=lambda path: path.stat().st_mtime)


explicit_aft = (
    AFT_WORKBOOK
    or os.environ.get("HEADWAY_AFT_WORKBOOK")
)
explicit_data = (
    DATA_WORKBOOK
    or os.environ.get("HEADWAY_DATA_WORKBOOK")
)
explicit_pooled = (
    POOLED_REFERENCE_WORKBOOK
    or os.environ.get("HEADWAY_POOLED_REFERENCE_WORKBOOK")
)

AFT_PATH = resolve_input(
    explicit_aft,
    is_aft_workbook,
    [
        "T5_41_Sequential_AFT_Selection_VIF.xlsx",
        "91690226-7193-4f85-95a0-d09d3419d581.xlsx",
    ],
)
DATA_PATH = resolve_input(
    explicit_data,
    is_data_workbook,
    ["data3.xlsx", "12-data3.xlsx"],
)
POOLED_REFERENCE_PATH = resolve_input(
    explicit_pooled,
    is_pooled_reference_workbook,
    ["T1_Baseline_Headway_Distributions.xlsx"],
)

if AFT_PATH is None:
    raise FileNotFoundError(
        "Could not find the 41-model sequential-AFT workbook. "
        "Set AFT_WORKBOOK to its path."
    )
if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find data3.xlsx. Set DATA_WORKBOOK to its path."
    )

AFT_SOURCE = pd.read_excel(
    AFT_PATH,
    sheet_name="S1_Model_summary",
)
DATA = pd.read_excel(DATA_PATH)

if len(AFT_SOURCE) != 41:
    raise AssertionError(
        f"Expected 41 final models; found {len(AFT_SOURCE)}"
    )
if AFT_SOURCE["Pair"].nunique() != 8:
    raise AssertionError(
        "Expected eight eligible target–leader pairs"
    )
if not REQUIRED_DATA_COLUMNS.issubset(DATA.columns):
    missing = sorted(REQUIRED_DATA_COLUMNS - set(DATA.columns))
    raise KeyError("Missing data columns: " + ", ".join(missing))

DATA[HEADWAY] = pd.to_numeric(DATA[HEADWAY], errors="coerce")
if DATA[HEADWAY].isna().any():
    raise ValueError("Time_Headway contains missing/non-numeric values")
if (DATA[HEADWAY] <= 0).any():
    raise ValueError("AFT headways must be strictly positive")

for covariate in CONTINUOUS_COVARIATES:
    DATA[covariate] = pd.to_numeric(
        DATA[covariate],
        errors="coerce",
    )
    if DATA[covariate].isna().any():
        raise ValueError(
            f"{covariate} contains missing/non-numeric values"
        )

print("AFT workbook       :", AFT_PATH)
print("Raw-data workbook  :", DATA_PATH)
print("Pooled reference   :", POOLED_REFERENCE_PATH)
print("Final bootstraps   :", N_BOOT_FINAL)
print("Pooled bootstraps  :", N_BOOT_POOLED)
print("Rows / pairs       :", len(DATA), "/", DATA["Pair"].nunique())


AFT workbook       : D:\Headway\final run\tables\T5_41_Sequential_AFT_Selection_VIF.xlsx
Raw-data workbook  : D:\Headway\data3.xlsx
Pooled reference   : D:\Headway\Tables\T1_Baseline_Headway_Distributions.xlsx
Final bootstraps   : 499
Pooled bootstraps  : 499
Rows / pairs       : 898 / 10


In [2]:
# ============================================================
# 2. Distribution definitions and multivariable AFT likelihood
# ============================================================

DIST_SPECS = {
    "gengamma": {
        "label": "Generalized gamma",
        "distribution": stats.gengamma,
        "mode": "floc0",
    },
    "weibull_min": {
        "label": "Weibull",
        "distribution": stats.weibull_min,
        "mode": "floc0",
    },
    "pearson3": {
        "label": "Pearson type III",
        "distribution": stats.pearson3,
        "mode": "free",
    },
    "gamma": {
        "label": "Gamma",
        "distribution": stats.gamma,
        "mode": "floc0",
    },
    "lognorm": {
        "label": "Log-normal",
        "distribution": stats.lognorm,
        "mode": "floc0",
    },
    "invgauss": {
        "label": "Inverse Gaussian",
        "distribution": stats.invgauss,
        "mode": "floc0",
    },
}


def fit_baseline(key, x):
    """Baseline MLE using the original 41-model conventions."""
    x = np.asarray(x, dtype=float)
    specification = DIST_SPECS[key]
    distribution = specification["distribution"]

    if specification["mode"] == "free":
        parameters = tuple(
            np.asarray(distribution.fit(x), dtype=float)
        )
        k = len(parameters)
    else:
        parameters = tuple(
            np.asarray(
                distribution.fit(x, floc=0.0),
                dtype=float,
            )
        )
        k = len(parameters) - 1

    log_likelihood = float(
        np.sum(distribution.logpdf(x, *parameters))
    )
    if not np.isfinite(log_likelihood):
        raise FloatingPointError(
            f"Non-finite baseline likelihood for {key}"
        )

    theta = theta_from_model(
        key,
        parameters,
        betas=np.empty(0),
    )

    return {
        "parameters": parameters,
        "betas": np.empty(0),
        "covariates": [],
        "theta": theta,
        "logLik": log_likelihood,
        "k": int(k),
        "AIC": float(2 * k - 2 * log_likelihood),
        "converged": True,
        "optimizer": "scipy.fit",
        "optimizer_message": "Baseline MLE",
    }


def base_parameter_count(key):
    if key in {"gengamma", "pearson3"}:
        return 3
    if key in {
        "weibull_min",
        "gamma",
        "lognorm",
        "invgauss",
    }:
        return 2
    raise KeyError(key)


def theta_from_model(key, parameters, betas):
    """Transform parameters to a stable unconstrained vector."""
    p = tuple(float(value) for value in parameters)
    betas = np.asarray(betas, dtype=float)

    if key == "gengamma":
        a, c, _, scale = p
        base = [
            np.log(a),
            np.log(abs(c)),
            np.log(scale),
        ]
    elif key in {
        "weibull_min",
        "gamma",
        "lognorm",
        "invgauss",
    }:
        shape, _, scale = p
        base = [np.log(shape), np.log(scale)]
    elif key == "pearson3":
        skew, loc, scale = p
        base = [skew, loc, np.log(scale)]
    else:
        raise KeyError(key)

    return np.concatenate(
        [np.asarray(base, dtype=float), betas]
    )


def model_from_theta(key, theta, n_covariates):
    """Return SciPy parameters and β vector from optimizer θ."""
    theta = np.asarray(theta, dtype=float)
    base_k = base_parameter_count(key)

    if len(theta) != base_k + n_covariates:
        raise ValueError("Optimizer-vector length mismatch")

    if key == "gengamma":
        parameters = (
            float(np.exp(theta[0])),
            float(np.exp(theta[1])),
            0.0,
            float(np.exp(theta[2])),
        )
    elif key in {
        "weibull_min",
        "gamma",
        "lognorm",
        "invgauss",
    }:
        parameters = (
            float(np.exp(theta[0])),
            0.0,
            float(np.exp(theta[1])),
        )
    elif key == "pearson3":
        parameters = (
            float(theta[0]),
            float(theta[1]),
            float(np.exp(theta[2])),
        )
    else:
        raise KeyError(key)

    betas = np.asarray(theta[base_k:], dtype=float)
    return parameters, betas


def optimizer_bounds(key, n_covariates):
    log_positive = (-12.0, 12.0)

    if key == "gengamma":
        base = [
            log_positive,
            log_positive,
            (-20.0, 20.0),
        ]
    elif key in {
        "weibull_min",
        "gamma",
        "lognorm",
        "invgauss",
    }:
        base = [log_positive, (-20.0, 20.0)]
    elif key == "pearson3":
        base = [
            (-50.0, 50.0),
            (None, None),
            (-20.0, 20.0),
        ]
    else:
        raise KeyError(key)

    return base + [(-3.0, 3.0)] * n_covariates


def aft_negative_loglik(theta, key, x, Z):
    """
    Conditional negative log-likelihood under:
        T_i = exp(Z_i @ beta) * Y_i.
    """
    try:
        Z = np.asarray(Z, dtype=float)
        parameters, betas = model_from_theta(
            key,
            theta,
            Z.shape[1],
        )
        eta = np.clip(Z @ betas, -50.0, 50.0)
        adjusted = x * np.exp(-eta)
        distribution = DIST_SPECS[key]["distribution"]
        log_density = (
            distribution.logpdf(adjusted, *parameters)
            - eta
        )
    except Exception:
        return 1e100

    if not np.all(np.isfinite(log_density)):
        return 1e100
    return float(-np.sum(log_density))


def map_start_model(key, covariate_names, start_model):
    """Map a nested or reduced fitted model into a new design."""
    beta_map = dict(
        zip(
            start_model.get("covariates", []),
            np.asarray(
                start_model.get("betas", []),
                dtype=float,
            ),
        )
    )
    betas = np.array(
        [beta_map.get(name, 0.0) for name in covariate_names],
        dtype=float,
    )
    return theta_from_model(
        key,
        start_model["parameters"],
        betas,
    )


def fit_aft_multi(
    key,
    x,
    Z,
    covariate_names,
    start_model=None,
    robust=True,
    allow_nonpositive=False,
):
    """Fit a zero- or multi-covariate AFT model by maximum likelihood."""
    x = np.asarray(x, dtype=float)
    Z = np.asarray(Z, dtype=float)
    covariate_names = list(covariate_names)

    if Z.ndim != 2 or Z.shape[0] != len(x):
        raise ValueError("Z must have shape n × p")
    if Z.shape[1] != len(covariate_names):
        raise ValueError("Design columns and names do not match")
    if not np.all(np.isfinite(x)) or not np.all(np.isfinite(Z)):
        raise ValueError("Non-finite model input")
    if np.any(x <= 0) and not allow_nonpositive:
        raise ValueError("Headways must be strictly positive")

    n_covariates = Z.shape[1]
    baseline = fit_baseline(key, x)
    if n_covariates == 0:
        return baseline

    null_theta = theta_from_model(
        key,
        baseline["parameters"],
        np.zeros(n_covariates),
    )

    candidate_starts = [
        ("null start", null_theta, True),
    ]

    if start_model is not None:
        nested_theta = map_start_model(
            key,
            covariate_names,
            start_model,
        )
        candidate_starts.insert(
            0,
            ("nested start", nested_theta, True),
        )

    if robust:
        # A log-linear least-squares estimate is only a starting value;
        # the reported fit always comes from the stated AFT likelihood.
        try:
            slopes = np.linalg.lstsq(
                np.column_stack(
                    [np.ones(len(x)), Z]
                ),
                np.log(x),
                rcond=None,
            )[0][1:]
            slopes = np.clip(slopes, -1.5, 1.5)

            for multiplier in [1.0, -1.0, 0.5]:
                theta = null_theta.copy()
                theta[-n_covariates:] = (
                    multiplier * slopes
                )
                candidate_starts.append(
                    (
                        f"log-linear start × {multiplier:g}",
                        theta,
                        False,
                    )
                )
        except Exception:
            pass

    # Deduplicate numerically identical starts.
    starts = []
    for label, theta, is_nested in candidate_starts:
        if not any(
            np.allclose(theta, existing[1], rtol=0, atol=1e-10)
            for existing in starts
        ):
            starts.append((label, theta, is_nested))

    candidates = []
    for label, theta, is_nested in starts:
        # Heuristic starts are optimizer initials only. They must not
        # be treated as fitted solutions before optimization.
        if not is_nested:
            continue
        objective = aft_negative_loglik(
            theta,
            key,
            x,
            Z,
        )
        if np.isfinite(objective):
            candidates.append(
                {
                    "theta": np.asarray(theta, dtype=float),
                    "fun": float(objective),
                    "success": bool(is_nested),
                    "message": label,
                    "method": label,
                }
            )

    bounds = optimizer_bounds(key, n_covariates)

    for label, initial, _ in starts:
        try:
            result = optimize.minimize(
                aft_negative_loglik,
                initial,
                args=(key, x, Z),
                method="L-BFGS-B",
                bounds=bounds,
                options={
                    "maxiter": 5000 if robust else 1200,
                    "ftol": 1e-10 if robust else 1e-8,
                    "gtol": 1e-7 if robust else 1e-5,
                    "maxls": 50,
                },
            )
        except Exception:
            continue

        if np.isfinite(result.fun):
            candidates.append(
                {
                    "theta": np.asarray(
                        result.x,
                        dtype=float,
                    ),
                    "fun": float(result.fun),
                    "success": bool(result.success),
                    "message": (
                        f"{label}: {result.message}"
                    ),
                    "method": "L-BFGS-B",
                }
            )

    if not candidates:
        raise RuntimeError("No finite optimizer candidate")

    best = min(candidates, key=lambda item: item["fun"])

    # One fallback for a better-but-nonconverged L-BFGS-B solution.
    if robust and not best["success"]:
        try:
            fallback = optimize.minimize(
                aft_negative_loglik,
                best["theta"],
                args=(key, x, Z),
                method="Powell",
                bounds=bounds,
                options={
                    "maxiter": 5000,
                    "xtol": 1e-7,
                    "ftol": 1e-9,
                },
            )
            if np.isfinite(fallback.fun):
                candidates.append(
                    {
                        "theta": np.asarray(
                            fallback.x,
                            dtype=float,
                        ),
                        "fun": float(fallback.fun),
                        "success": bool(fallback.success),
                        "message": str(fallback.message),
                        "method": "Powell",
                    }
                )
                best = min(
                    candidates,
                    key=lambda item: item["fun"],
                )
        except Exception:
            pass

    parameters, betas = model_from_theta(
        key,
        best["theta"],
        n_covariates,
    )
    log_likelihood = float(-best["fun"])
    k = baseline["k"] + n_covariates

    return {
        "parameters": parameters,
        "betas": betas,
        "covariates": covariate_names,
        "theta": np.asarray(best["theta"], dtype=float),
        "logLik": log_likelihood,
        "k": int(k),
        "AIC": float(2 * k - 2 * log_likelihood),
        "converged": bool(best["success"]),
        "optimizer": best["method"],
        "optimizer_message": best["message"],
    }


def nested_lr_test(full_model, reduced_model, df_difference=1):
    """Likelihood-ratio comparison of two nested AFT models."""
    statistic = max(
        0.0,
        2.0
        * (
            float(full_model["logLik"])
            - float(reduced_model["logLik"])
        ),
    )
    p_value = float(
        stats.chi2.sf(statistic, df=df_difference)
    )
    return statistic, p_value


In [3]:
# ============================================================
# 3. Reconstruct final designs and refit all 41 AFT models
# ============================================================

def encode_covariate(pair_data, covariate):
    """
    Reproduce the coding used in the sequential AFT notebook.

    Continuous predictors are standardized within the pair.
    """
    series = pair_data[covariate]

    if covariate in CONTINUOUS_COVARIATES:
        values = pd.to_numeric(
            series,
            errors="coerce",
        ).to_numpy(float)
        mean = float(np.mean(values))
        sd = float(np.std(values, ddof=1))
        if not np.isfinite(sd) or sd <= 1e-12:
            raise ValueError(
                f"{covariate} is constant within the pair"
            )
        return (
            (values - mean) / sd,
            (
                f"Within-pair z-score; "
                f"mean={mean:.6g}, SD={sd:.6g}"
            ),
        )

    if covariate in {"Occupancy", "Off_centeredness"}:
        if pd.api.types.is_bool_dtype(series):
            values = series.astype(float).to_numpy()
        else:
            normalized = (
                series.astype(str)
                .str.strip()
                .str.lower()
            )
            mapping = {
                "true": 1.0,
                "false": 0.0,
                "yes": 1.0,
                "no": 0.0,
                "1": 1.0,
                "0": 0.0,
            }
            values = normalized.map(mapping).to_numpy(float)
        if not np.all(np.isfinite(values)):
            raise ValueError(
                f"Unrecognized binary coding in {covariate}"
            )
        return values, "False=0, True=1"

    if covariate == "Site":
        values = (
            series.astype(str).str.strip() == SITE_ONE_LEVEL
        ).astype(float).to_numpy()
        return (
            values,
            f"Shahjahanpur=0, {SITE_ONE_LEVEL}=1",
        )

    raise KeyError(covariate)


def build_pair_design(pair_data):
    design = {}
    coding = {}
    for covariate in COVARIATES:
        values, note = encode_covariate(
            pair_data,
            covariate,
        )
        design[covariate] = np.asarray(values, dtype=float)
        coding[covariate] = note
    return design, coding


def design_matrix(design, covariates):
    covariates = list(covariates)
    if not covariates:
        n = len(next(iter(design.values())))
        return np.empty((n, 0), dtype=float)
    return np.column_stack(
        [design[covariate] for covariate in covariates]
    )


def parse_covariates(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text or text.lower() in {
        "(none)",
        "none",
        "nan",
    }:
        return []
    covariates = [
        item.strip()
        for item in text.split(",")
        if item.strip()
    ]
    unknown = [
        item for item in covariates
        if item not in COVARIATES
    ]
    if unknown:
        raise KeyError(
            "Unknown final covariates: " + ", ".join(unknown)
        )
    return covariates


def aicc(aic, k, n):
    denominator = n - k - 1
    if denominator <= 0:
        return np.inf
    return float(
        aic + 2 * k * (k + 1) / denominator
    )


def positive_support_mass(key, fitted_model):
    distribution = DIST_SPECS[key]["distribution"]
    mass = distribution.cdf(
        0.0,
        *fitted_model["parameters"],
    )
    return float(mass) if np.isfinite(mass) else np.nan


fitted_41 = []
audit_rows = []

for model_number, source_row in enumerate(
    AFT_SOURCE.to_dict("records"),
    start=1,
):
    pair = source_row["Pair"]
    distribution = source_row["Distribution"]
    key = source_row["Distribution key"]
    covariates = parse_covariates(
        source_row["Final covariates"]
    )

    pair_data = (
        DATA.loc[DATA["Pair"].eq(pair)]
        .copy()
        .reset_index(drop=True)
    )
    x = pair_data[HEADWAY].to_numpy(float)
    design, coding = build_pair_design(pair_data)
    # Reconstruct the recorded final model through its retained-term
    # order. Using each nested fit as the next optimizer start exactly
    # mirrors the sequential notebook and is important for flexible
    # families with multiple local optima.
    fitted = fit_baseline(key, x)
    for step_number in range(1, len(covariates) + 1):
        step_covariates = covariates[:step_number]
        Z_step = design_matrix(
            design,
            step_covariates,
        )
        fitted = fit_aft_multi(
            key,
            x,
            Z_step,
            step_covariates,
            start_model=fitted,
            robust=True,
            allow_nonpositive=False,
        )

    Z = design_matrix(design, covariates)

    reference_aic = float(source_row["Final AIC"])
    difference = fitted["AIC"] - reference_aic
    reconciled = abs(difference) <= AIC_AUDIT_TOLERANCE
    support_mass = positive_support_mass(key, fitted)

    fitted_41.append(
        {
            "model_order": model_number,
            "pair": pair,
            "target": pair.split("_following_")[0],
            "distribution": distribution,
            "key": key,
            "covariates": covariates,
            "pair_data": pair_data,
            "x": x,
            "design": design,
            "coding": coding,
            "Z": Z,
            "fit": fitted,
            "reference_row": source_row,
            "reference_aic": reference_aic,
            "aic_difference": difference,
            "aic_reconciled": reconciled,
            "nonpositive_mass": support_mass,
        }
    )

    audit_rows.append(
        {
            "Model order": model_number,
            "Pair": pair,
            "Distribution": distribution,
            "Distribution key": key,
            "n": len(x),
            "Final covariates": (
                ", ".join(covariates)
                if covariates
                else "(none)"
            ),
            "Reference final logLik": source_row["Final logLik"],
            "Refitted final logLik": fitted["logLik"],
            "Reference final AIC": reference_aic,
            "Refitted final AIC": fitted["AIC"],
            "AIC difference": difference,
            "AIC reconciled": reconciled,
            "Final k": fitted["k"],
            "Final AICc (reported for sensitivity)": aicc(
                fitted["AIC"],
                fitted["k"],
                len(x),
            ),
            "Optimizer converged": fitted["converged"],
            "Optimizer": fitted["optimizer"],
            "Baseline probability mass at T0 ≤ 0": support_mass,
            "Positive-support requirement met": (
                np.isfinite(support_mass)
                and support_mass
                <= POSITIVE_SUPPORT_TOLERANCE
            ),
        }
    )

    print(
        f"[Refit {model_number:02d}/41] "
        f"{pair} | {distribution} | "
        f"Δaudit={difference:+.3e}"
    )

AFT_AUDIT = pd.DataFrame(audit_rows)

if not AFT_AUDIT["AIC reconciled"].all():
    failed = AFT_AUDIT.loc[
        ~AFT_AUDIT["AIC reconciled"],
        [
            "Pair",
            "Distribution",
            "AIC difference",
        ],
    ]
    raise AssertionError(
        "At least one final AFT AIC did not reconcile:\n"
        + failed.to_string(index=False)
    )

print(
    "\nAll 41 final AFT AIC values reconciled within",
    AIC_AUDIT_TOLERANCE,
)


[Refit 01/41] PR_following_4W | Weibull | Δaudit=-2.842e-14
[Refit 02/41] PR_following_4W | Generalized gamma | Δaudit=-2.842e-14
[Refit 03/41] PR_following_4W | Pearson type III | Δaudit=+2.842e-14
[Refit 04/41] PR_following_4W | Gamma | Δaudit=+2.842e-14
[Refit 05/41] PR_following_4W | Log-normal | Δaudit=-2.842e-14
[Refit 06/41] PR_following_4W | Inverse Gaussian | Δaudit=+0.000e+00
[Refit 07/41] PR_following_MT_3W | Weibull | Δaudit=+0.000e+00
[Refit 08/41] PR_following_MT_3W | Generalized gamma | Δaudit=+0.000e+00
[Refit 09/41] PR_following_MT_3W | Pearson type III | Δaudit=+0.000e+00
[Refit 10/41] PR_following_MT_3W | Gamma | Δaudit=+0.000e+00
[Refit 11/41] PR_following_NMT_3W | Weibull | Δaudit=+2.842e-14
[Refit 12/41] PR_following_NMT_3W | Generalized gamma | Δaudit=+0.000e+00
[Refit 13/41] PR_following_NMT_3W | Pearson type III | Δaudit=+0.000e+00
[Refit 14/41] PR_following_NMT_3W | Gamma | Δaudit=-2.842e-14
[Refit 15/41] BTW_following_4W | Generalized gamma | Δaudit=+0.000e+0

In [4]:
# ============================================================
# 4. Rank final AFT models and form the within-pair evidence set
# ============================================================

ranking_rows = []
fit_lookup = {}

for item in fitted_41:
    pair = item["pair"]
    distribution = item["distribution"]
    fitted = item["fit"]
    fit_lookup[(pair, distribution)] = item

    ranking_rows.append(
        {
            "Model order": item["model_order"],
            "Pair": pair,
            "Target": item["target"],
            "Distribution": distribution,
            "Distribution key": item["key"],
            "n": len(item["x"]),
            "Final covariates": (
                ", ".join(item["covariates"])
                if item["covariates"]
                else "(none)"
            ),
            "k": fitted["k"],
            "logLik": fitted["logLik"],
            "AIC": fitted["AIC"],
            "AICc": aicc(
                fitted["AIC"],
                fitted["k"],
                len(item["x"]),
            ),
            "Baseline probability mass at T0 ≤ 0": (
                item["nonpositive_mass"]
            ),
            "Positive-support requirement met": (
                np.isfinite(item["nonpositive_mass"])
                and item["nonpositive_mass"]
                <= POSITIVE_SUPPORT_TOLERANCE
            ),
        }
    )

AFT_RANKED = pd.DataFrame(ranking_rows)
AFT_RANKED["ΔAIC"] = (
    AFT_RANKED["AIC"]
    - AFT_RANKED.groupby("Pair")["AIC"].transform("min")
)
AFT_RANKED["Relative likelihood"] = np.exp(
    -0.5 * AFT_RANKED["ΔAIC"]
)
AFT_RANKED["Akaike weight"] = (
    AFT_RANKED["Relative likelihood"]
    / AFT_RANKED.groupby("Pair")[
        "Relative likelihood"
    ].transform("sum")
)
AFT_RANKED["AIC rank"] = (
    AFT_RANKED.groupby("Pair")["AIC"]
    .rank(method="first")
    .astype(int)
)

if CUTOFF_IS_INCLUSIVE:
    in_evidence_set = AFT_RANKED["ΔAIC"].le(
        DELTA_AIC_CUTOFF
    )
    cutoff_text = f"ΔAIC ≤ {DELTA_AIC_CUTOFF:g}"
else:
    in_evidence_set = AFT_RANKED["ΔAIC"].lt(
        DELTA_AIC_CUTOFF
    )
    cutoff_text = f"ΔAIC < {DELTA_AIC_CUTOFF:g}"

# The best model is always included even if floating-point noise occurs.
AFT_RANKED["Retained for final bootstrap"] = (
    in_evidence_set
    | AFT_RANKED["AIC rank"].eq(1)
)
AFT_RANKED["Fit role"] = np.where(
    AFT_RANKED["AIC rank"].eq(1),
    "Best",
    np.where(
        AFT_RANKED["Retained for final bootstrap"],
        "Competing",
        "Outside final evidence set",
    ),
)

AFT_RANKED = AFT_RANKED.sort_values(
    ["Pair", "AIC", "Distribution"],
    ignore_index=True,
)
AFT_EVIDENCE_SET = AFT_RANKED.loc[
    AFT_RANKED["Retained for final bootstrap"]
].copy()

if FINAL_MODEL_LIMIT > 0:
    AFT_MODELS_TO_BOOTSTRAP = AFT_EVIDENCE_SET.head(
        FINAL_MODEL_LIMIT
    ).copy()
else:
    AFT_MODELS_TO_BOOTSTRAP = AFT_EVIDENCE_SET.copy()

print("Final AFT evidence rule :", cutoff_text)
print("Pairs                   :", AFT_EVIDENCE_SET["Pair"].nunique())
print("Models retained          :", len(AFT_EVIDENCE_SET))
display(
    AFT_EVIDENCE_SET[
        [
            "Pair",
            "Distribution",
            "Final covariates",
            "AIC",
            "ΔAIC",
            "Akaike weight",
            "Fit role",
        ]
    ].round(
        {
            "AIC": 3,
            "ΔAIC": 3,
            "Akaike weight": 3,
        }
    )
)


Final AFT evidence rule : ΔAIC < 2
Pairs                   : 8
Models retained          : 17


,Pair,Distribution,Final covariates,AIC,ΔAIC,Akaike weight,Fit role
0,BTW_following_4W,Generalized gamma,"Speed_Difference, Target_Speed_km/hr, Off_cent...",564.886,0.000,0.585,Best
1,BTW_following_4W,Pearson type III,"Speed_Difference, Target_Speed_km/hr, Off_cent...",565.574,0.688,0.415,Competing
4,BTW_following_MT_2W,Gamma,"Speed_Difference, Target_Speed_km/hr",158.646,0.000,0.659,Best
10,BTW_following_MT_3W,Inverse Gaussian,"Target_Speed_km/hr, Speed_Difference",381.460,0.000,0.490,Best
11,BTW_following_MT_3W,Log-normal,"Target_Speed_km/hr, Speed_Difference",381.850,0.390,0.403,Competing
15,BTW_following_NMT_2W,Inverse Gaussian,Speed_Difference,86.484,0.000,0.340,Best
16,BTW_following_NMT_2W,Log-normal,Speed_Difference,87.043,0.559,0.257,Competing
17,BTW_following_NMT_2W,Gamma,Speed_Difference,87.815,1.331,0.175,Competing
21,BTW_following_NMT_3W,Gamma,Target_Speed_km/hr,293.911,0.000,0.305,Best
22,BTW_following_NMT_3W,Pearson type III,Target_Speed_km/hr,294.390,0.478,0.240,Competing


In [5]:
# ============================================================
# 5. Conditional PIT statistics and refitted parametric bootstrap
# ============================================================

def uniform_ks_ad(u):
    """KS D and AD A² for testing a sample against Uniform(0,1)."""
    u = np.sort(
        np.clip(
            np.asarray(u, dtype=float),
            1e-12,
            1 - 1e-12,
        )
    )
    n = len(u)
    i = np.arange(1, n + 1)

    ks_d = max(
        np.max(i / n - u),
        np.max(u - (i - 1) / n),
    )
    ad_a2 = (
        -n
        - np.sum(
            (2 * i - 1)
            * (
                np.log(u)
                + np.log(1 - u[::-1])
            )
        )
        / n
    )
    return float(ks_d), float(ad_a2)


def conditional_pit(key, x, Z, fitted_model):
    x = np.asarray(x, dtype=float)
    Z = np.asarray(Z, dtype=float)
    betas = np.asarray(fitted_model["betas"], dtype=float)
    eta = (
        np.clip(Z @ betas, -50.0, 50.0)
        if Z.shape[1]
        else np.zeros(len(x), dtype=float)
    )
    adjusted = x * np.exp(-eta)
    distribution = DIST_SPECS[key]["distribution"]
    u = distribution.cdf(
        adjusted,
        *fitted_model["parameters"],
    )
    if not np.all(np.isfinite(u)):
        raise FloatingPointError(
            "Non-finite conditional CDF values"
        )
    return np.asarray(u, dtype=float)


def bootstrap_conditional_gof(
    key,
    x,
    Z,
    covariate_names,
    fitted_model,
    n_boot,
    seed,
):
    """
    Parametric-bootstrap KS and AD tests for a fitted AFT model.

    The observed covariate matrix remains fixed. In every replicate:
      1. draw baseline T0 values from the fitted family;
      2. form T = exp(Zβ)T0;
      3. refit all baseline parameters and β values;
      4. recompute conditional PIT KS and AD statistics.
    """
    x = np.asarray(x, dtype=float)
    Z = np.asarray(Z, dtype=float)
    n = len(x)
    distribution = DIST_SPECS[key]["distribution"]

    observed_u = conditional_pit(
        key,
        x,
        Z,
        fitted_model,
    )
    observed_ks, observed_ad = uniform_ks_ad(observed_u)

    if n_boot <= 0:
        return {
            "KS D": observed_ks,
            "KS p": np.nan,
            "AD A²": observed_ad,
            "AD p": np.nan,
            "Bootstrap KS exceedances": 0,
            "Bootstrap AD exceedances": 0,
            "Bootstrap valid": 0,
            "Bootstrap requested": 0,
            "Bootstrap status": "Not requested",
        }

    rng = np.random.default_rng(seed)
    betas = np.asarray(fitted_model["betas"], dtype=float)
    eta = (
        np.clip(Z @ betas, -50.0, 50.0)
        if Z.shape[1]
        else np.zeros(n, dtype=float)
    )

    ks_exceedances = 0
    ad_exceedances = 0
    valid = 0

    for _ in range(n_boot):
        try:
            baseline_draw = np.asarray(
                distribution.rvs(
                    *fitted_model["parameters"],
                    size=n,
                    random_state=rng,
                ),
                dtype=float,
            )
            if (
                baseline_draw.shape != (n,)
                or not np.all(np.isfinite(baseline_draw))
                or np.any(baseline_draw <= 0)
            ):
                continue

            simulated_x = baseline_draw * np.exp(eta)
            if (
                not np.all(np.isfinite(simulated_x))
                or np.any(simulated_x <= 0)
            ):
                continue

            refit = fit_aft_multi(
                key,
                simulated_x,
                Z,
                covariate_names,
                start_model=fitted_model,
                robust=False,
                allow_nonpositive=False,
            )
            bootstrap_u = conditional_pit(
                key,
                simulated_x,
                Z,
                refit,
            )
            bootstrap_ks, bootstrap_ad = uniform_ks_ad(
                bootstrap_u
            )
            if not (
                np.isfinite(bootstrap_ks)
                and np.isfinite(bootstrap_ad)
            ):
                continue
        except Exception:
            continue

        valid += 1
        ks_exceedances += int(
            bootstrap_ks >= observed_ks
        )
        ad_exceedances += int(
            bootstrap_ad >= observed_ad
        )

    minimum_valid = int(
        np.ceil(MIN_VALID_BOOT_FRACTION * n_boot)
    )
    if valid < minimum_valid:
        return {
            "KS D": observed_ks,
            "KS p": np.nan,
            "AD A²": observed_ad,
            "AD p": np.nan,
            "Bootstrap KS exceedances": ks_exceedances,
            "Bootstrap AD exceedances": ad_exceedances,
            "Bootstrap valid": valid,
            "Bootstrap requested": n_boot,
            "Bootstrap status": (
                "INSUFFICIENT VALID REPLICATES: "
                f"{valid}/{n_boot}"
            ),
        }

    return {
        "KS D": observed_ks,
        "KS p": float(
            (1 + ks_exceedances) / (1 + valid)
        ),
        "AD A²": observed_ad,
        "AD p": float(
            (1 + ad_exceedances) / (1 + valid)
        ),
        "Bootstrap KS exceedances": ks_exceedances,
        "Bootstrap AD exceedances": ad_exceedances,
        "Bootstrap valid": valid,
        "Bootstrap requested": n_boot,
        "Bootstrap status": "OK",
    }


In [6]:
# ============================================================
# 6. Cox–Snell residuals, Nelson–Aalen curves, and envelopes
# ============================================================

def cox_snell_residuals(key, x, Z, fitted_model):
    x = np.asarray(x, dtype=float)
    Z = np.asarray(Z, dtype=float)
    betas = np.asarray(fitted_model["betas"], dtype=float)
    eta = (
        np.clip(Z @ betas, -50.0, 50.0)
        if Z.shape[1]
        else np.zeros(len(x), dtype=float)
    )
    adjusted = x * np.exp(-eta)
    distribution = DIST_SPECS[key]["distribution"]

    # sf is numerically more stable than 1 - cdf in the upper tail.
    survival = distribution.sf(
        adjusted,
        *fitted_model["parameters"],
    )
    survival = np.clip(
        np.asarray(survival, dtype=float),
        1e-300,
        1.0,
    )
    residuals = -np.log(survival)
    if not np.all(np.isfinite(residuals)):
        raise FloatingPointError(
            "Non-finite Cox–Snell residuals"
        )
    return residuals


def nelson_aalen_all_events(residuals):
    """
    Nelson–Aalen estimate with all observations treated as events.

    Ties are handled as d/Y at each distinct residual value.
    """
    residuals = np.asarray(residuals, dtype=float)
    values, counts = np.unique(
        np.sort(residuals),
        return_counts=True,
    )
    n = len(residuals)
    at_risk = n - np.concatenate(
        [[0], np.cumsum(counts[:-1])]
    )
    increments = counts / at_risk
    cumulative_hazard = np.cumsum(increments)
    return values, cumulative_hazard


def cox_snell_envelope(n, x_grid, simulations, seed):
    """
    Pointwise simulation envelope for a unit-exponential sample.

    This is a visual aid, not an additional formal acceptance test.
    """
    if simulations <= 0:
        return (
            np.full_like(x_grid, np.nan, dtype=float),
            np.full_like(x_grid, np.nan, dtype=float),
        )

    rng = np.random.default_rng(seed)
    curves = np.empty(
        (simulations, len(x_grid)),
        dtype=float,
    )

    for index in range(simulations):
        simulated = rng.exponential(scale=1.0, size=n)
        x_sim, h_sim = nelson_aalen_all_events(simulated)
        curves[index] = np.interp(
            x_grid,
            x_sim,
            h_sim,
            left=0.0,
            right=h_sim[-1],
        )

    return (
        np.quantile(curves, 0.025, axis=0),
        np.quantile(curves, 0.975, axis=0),
    )


def cox_snell_metrics(residuals):
    residuals = np.asarray(residuals, dtype=float)
    x_na, h_na = nelson_aalen_all_events(residuals)

    central_limit = float(
        np.quantile(
            residuals,
            min(COX_SNELL_PLOT_QUANTILE, 0.95),
        )
    )
    central = x_na <= central_limit
    if not central.any():
        central = np.ones_like(x_na, dtype=bool)

    x_central = x_na[central]
    h_central = h_na[central]
    denominator = float(np.sum(x_central**2))
    slope = (
        float(np.sum(x_central * h_central) / denominator)
        if denominator > 0
        else np.nan
    )

    return {
        "Residual mean (ideal 1)": float(np.mean(residuals)),
        "Residual variance (ideal 1)": float(
            np.var(residuals, ddof=1)
        ),
        "Central Nelson–Aalen slope (ideal 1)": slope,
        "Central RMSE from H(r)=r": float(
            np.sqrt(np.mean((h_central - x_central) ** 2))
        ),
        "Central max |H(r)-r|": float(
            np.max(np.abs(h_central - x_central))
        ),
        "Maximum residual": float(np.max(residuals)),
        "Nelson–Aalen H at maximum residual": float(h_na[-1]),
    }


def safe_filename(text):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(text)).strip("_")


def draw_cox_snell_panel(ax, item, residuals):
    x_na, h_na = nelson_aalen_all_events(residuals)
    raw_limit = float(
        np.quantile(residuals, COX_SNELL_PLOT_QUANTILE)
    )
    x_limit = min(
        COX_SNELL_MAX_X,
        max(1.0, raw_limit),
    )
    x_grid = np.linspace(0.0, x_limit, 240)
    low, high = cox_snell_envelope(
        len(residuals),
        x_grid,
        COX_SNELL_ENVELOPE_SIMULATIONS,
        stable_seed(
            item["pair"],
            item["distribution"],
            "cox-envelope",
        ),
    )

    if np.all(np.isfinite(low)) and np.all(np.isfinite(high)):
        ax.fill_between(
            x_grid,
            low,
            high,
            color="#D9EAF7",
            alpha=0.75,
            label="95% simulation envelope",
        )

    ax.step(
        x_na,
        h_na,
        where="post",
        color="#1F4E78",
        linewidth=1.8,
        label="Nelson–Aalen",
    )
    ax.plot(
        [0, x_limit],
        [0, x_limit],
        linestyle="--",
        color="#C00000",
        linewidth=1.3,
        label="H(r)=r",
    )
    ax.set_xlim(0, x_limit)
    ax.set_ylim(0, x_limit)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=0.25)
    ax.set_xlabel("Cox–Snell residual, r")
    ax.set_ylabel("Nelson–Aalen cumulative hazard")
    covariate_text = (
        ", ".join(
            COVARIATE_LABELS.get(value, value)
            for value in item["covariates"]
        )
        if item["covariates"]
        else "(none)"
    )
    covariate_text = "\n".join(
        textwrap.wrap(
            covariate_text,
            width=48,
            break_long_words=False,
            break_on_hyphens=False,
        )
    )
    ax.set_title(
        f"{item['distribution']}\n{covariate_text}",
        fontsize=10,
    )

    beyond = int(np.sum(residuals > x_limit))
    if beyond:
        ax.text(
            0.98,
            0.04,
            f"{beyond} residual(s) beyond plotted range",
            transform=ax.transAxes,
            ha="right",
            va="bottom",
            fontsize=8,
            color="#7F6000",
        )


In [7]:
# ============================================================
# 7. Bootstrap the final AFT evidence set and create diagnostics
# ============================================================

aft_gof_rows = []
cox_rows = []
cox_payload = {}

bootstrap_start = time.time()

for model_number, row in enumerate(
    AFT_MODELS_TO_BOOTSTRAP.to_dict("records"),
    start=1,
):
    item = fit_lookup[(row["Pair"], row["Distribution"])]
    print(
        f"[AFT bootstrap {model_number}/"
        f"{len(AFT_MODELS_TO_BOOTSTRAP)}] "
        f"{item['pair']} | {item['distribution']}",
        end="",
    )
    model_start = time.time()

    gof = bootstrap_conditional_gof(
        item["key"],
        item["x"],
        item["Z"],
        item["covariates"],
        item["fit"],
        n_boot=N_BOOT_FINAL,
        seed=stable_seed(
            item["pair"],
            item["distribution"],
            "final-aft-bootstrap",
        ),
    )

    ks_accepted = (
        bool(gof["KS p"] >= ALPHA_GOF)
        if np.isfinite(gof["KS p"])
        else pd.NA
    )
    ad_accepted = (
        bool(gof["AD p"] >= ALPHA_GOF)
        if np.isfinite(gof["AD p"])
        else pd.NA
    )
    accepted_both = (
        bool(ks_accepted and ad_accepted)
        if ks_accepted is not pd.NA
        and ad_accepted is not pd.NA
        else pd.NA
    )

    aft_gof_rows.append(
        {
            "Pair": item["pair"],
            "Target": item["target"],
            "Distribution": item["distribution"],
            "Distribution key": item["key"],
            "Fit role": row["Fit role"],
            "Final covariates": row["Final covariates"],
            "n": row["n"],
            "k": row["k"],
            "logLik": row["logLik"],
            "AIC": row["AIC"],
            "AICc": row["AICc"],
            "ΔAIC": row["ΔAIC"],
            "Akaike weight": row["Akaike weight"],
            "KS D": gof["KS D"],
            "KS bootstrap p": gof["KS p"],
            "KS accepted": ks_accepted,
            "AD A²": gof["AD A²"],
            "AD bootstrap p": gof["AD p"],
            "AD accepted": ad_accepted,
            "Accepted by both tests": accepted_both,
            "Bootstrap KS exceedances": gof[
                "Bootstrap KS exceedances"
            ],
            "Bootstrap AD exceedances": gof[
                "Bootstrap AD exceedances"
            ],
            "Bootstrap valid": gof["Bootstrap valid"],
            "Bootstrap requested": gof["Bootstrap requested"],
            "Bootstrap status": gof["Bootstrap status"],
            "Baseline probability mass at T0 ≤ 0": (
                item["nonpositive_mass"]
            ),
            "Positive-support requirement met": (
                np.isfinite(item["nonpositive_mass"])
                and item["nonpositive_mass"]
                <= POSITIVE_SUPPORT_TOLERANCE
            ),
        }
    )

    residuals = cox_snell_residuals(
        item["key"],
        item["x"],
        item["Z"],
        item["fit"],
    )
    metrics = cox_snell_metrics(residuals)
    cox_rows.append(
        {
            "Pair": item["pair"],
            "Distribution": item["distribution"],
            "Final covariates": row["Final covariates"],
            "n": len(residuals),
            **metrics,
        }
    )
    cox_payload[
        (item["pair"], item["distribution"])
    ] = {
        "item": item,
        "residuals": residuals,
    }

    # Save one high-resolution plot per selected model.
    figure, axis = plt.subplots(figsize=(6.4, 5.6))
    draw_cox_snell_panel(axis, item, residuals)
    handles, labels = axis.get_legend_handles_labels()
    if handles:
        axis.legend(
            handles,
            labels,
            loc="upper left",
            fontsize=8,
            frameon=True,
        )
    figure.suptitle(
        item["pair"].replace("_following_", " → "),
        fontsize=12,
        fontweight="bold",
    )
    figure.tight_layout()
    plot_path = INDIVIDUAL_FIGURES_DIR / (
        safe_filename(
            f"{item['pair']}__{item['distribution']}"
        )
        + ".png"
    )
    figure.savefig(plot_path, dpi=220, bbox_inches="tight")
    plt.close(figure)

    print(
        f" | valid={gof['Bootstrap valid']}/"
        f"{gof['Bootstrap requested']}"
        f" | {time.time() - model_start:.1f}s"
    )

AFT_FINAL_GOF = pd.DataFrame(aft_gof_rows)
COX_SNELL_METRICS = pd.DataFrame(cox_rows)


# Multipage PDF: one page per pair, with all ΔAIC<2 candidates.
with PdfPages(OUTPUT_PDF) as pdf:
    for pair in AFT_EVIDENCE_SET["Pair"].drop_duplicates():
        pair_rows = AFT_EVIDENCE_SET.loc[
            AFT_EVIDENCE_SET["Pair"].eq(pair)
        ]
        pair_rows = pair_rows.loc[
            pair_rows.apply(
                lambda row: (
                    (row["Pair"], row["Distribution"])
                    in cox_payload
                ),
                axis=1,
            )
        ]
        if pair_rows.empty:
            continue

        n_panels = len(pair_rows)
        n_columns = 2 if n_panels > 1 else 1
        n_rows = int(math.ceil(n_panels / n_columns))
        figure, axes = plt.subplots(
            n_rows,
            n_columns,
            figsize=(7.2 * n_columns, 5.8 * n_rows),
            squeeze=False,
        )
        axes_flat = axes.ravel()

        for axis, row in zip(
            axes_flat,
            pair_rows.to_dict("records"),
        ):
            payload = cox_payload[
                (row["Pair"], row["Distribution"])
            ]
            draw_cox_snell_panel(
                axis,
                payload["item"],
                payload["residuals"],
            )

        for axis in axes_flat[n_panels:]:
            axis.axis("off")

        handles, labels = axes_flat[0].get_legend_handles_labels()
        if handles:
            figure.legend(
                handles,
                labels,
                loc="lower center",
                ncol=min(3, len(handles)),
                frameon=False,
            )

        figure.suptitle(
            (
                "Cox–Snell diagnostics: "
                + pair.replace("_following_", " → ")
            ),
            fontsize=15,
            fontweight="bold",
        )
        figure.tight_layout(rect=[0, 0.05, 1, 0.96])
        pdf.savefig(figure, bbox_inches="tight")
        plt.close(figure)

print(
    "\nAFT bootstrap elapsed:",
    f"{time.time() - bootstrap_start:.1f} seconds",
)
print("Cox–Snell PDF:", OUTPUT_PDF)


[AFT bootstrap 1/17] BTW_following_4W | Generalized gamma | valid=499/499 | 34.9s
[AFT bootstrap 2/17] BTW_following_4W | Pearson type III | valid=484/499 | 20.4s
[AFT bootstrap 3/17] BTW_following_MT_2W | Gamma | valid=499/499 | 6.0s
[AFT bootstrap 4/17] BTW_following_MT_3W | Inverse Gaussian | valid=499/499 | 7.0s
[AFT bootstrap 5/17] BTW_following_MT_3W | Log-normal | valid=499/499 | 6.2s
[AFT bootstrap 6/17] BTW_following_NMT_2W | Inverse Gaussian | valid=499/499 | 5.1s
[AFT bootstrap 7/17] BTW_following_NMT_2W | Log-normal | valid=499/499 | 4.4s
[AFT bootstrap 8/17] BTW_following_NMT_2W | Gamma | valid=499/499 | 4.9s
[AFT bootstrap 9/17] BTW_following_NMT_3W | Gamma | valid=499/499 | 4.6s
[AFT bootstrap 10/17] BTW_following_NMT_3W | Pearson type III | valid=499/499 | 8.4s
[AFT bootstrap 11/17] BTW_following_NMT_3W | Inverse Gaussian | valid=499/499 | 5.1s
[AFT bootstrap 12/17] BTW_following_NMT_3W | Generalized gamma | valid=499/499 | 38.8s
[AFT bootstrap 13/17] PR_following_4W | 

In [8]:
# ============================================================
# 8. Fit and bootstrap comparable pooled distributions
# ============================================================

POOLED_GROUPS = {
    "All": np.ones(len(DATA), dtype=bool),
    "PR": DATA["V_Target"].astype(str).eq("PR").to_numpy(),
    "BTW": DATA["V_Target"].astype(str).eq("BTW").to_numpy(),
}

# Use exactly the family set present in the final pairwise inventory.
family_inventory = (
    AFT_SOURCE[
        ["Distribution key", "Distribution"]
    ]
    .drop_duplicates()
    .set_index("Distribution key")["Distribution"]
    .to_dict()
)

pooled_fits = {}
pooled_ranking_rows = []

for group, mask in POOLED_GROUPS.items():
    x = DATA.loc[mask, HEADWAY].to_numpy(float)
    for key, distribution in family_inventory.items():
        fitted = fit_baseline(key, x)
        support_mass = positive_support_mass(key, fitted)
        pooled_fits[(group, distribution)] = {
            "group": group,
            "distribution": distribution,
            "key": key,
            "x": x,
            "Z": np.empty((len(x), 0), dtype=float),
            "covariates": [],
            "fit": fitted,
            "nonpositive_mass": support_mass,
        }
        pooled_ranking_rows.append(
            {
                "Pooled group": group,
                "Distribution": distribution,
                "Distribution key": key,
                "n": len(x),
                "k": fitted["k"],
                "logLik": fitted["logLik"],
                "AIC": fitted["AIC"],
                "AICc": aicc(
                    fitted["AIC"],
                    fitted["k"],
                    len(x),
                ),
                "Baseline probability mass at T0 ≤ 0": (
                    support_mass
                ),
                "Positive-support requirement met": (
                    np.isfinite(support_mass)
                    and support_mass
                    <= POSITIVE_SUPPORT_TOLERANCE
                ),
            }
        )

POOLED_RANKED = pd.DataFrame(pooled_ranking_rows)
POOLED_RANKED["ΔAIC"] = (
    POOLED_RANKED["AIC"]
    - POOLED_RANKED.groupby("Pooled group")[
        "AIC"
    ].transform("min")
)
POOLED_RANKED["Relative likelihood"] = np.exp(
    -0.5 * POOLED_RANKED["ΔAIC"]
)
POOLED_RANKED["Akaike weight"] = (
    POOLED_RANKED["Relative likelihood"]
    / POOLED_RANKED.groupby("Pooled group")[
        "Relative likelihood"
    ].transform("sum")
)
POOLED_RANKED["AIC rank"] = (
    POOLED_RANKED.groupby("Pooled group")["AIC"]
    .rank(method="first")
    .astype(int)
)

if CUTOFF_IS_INCLUSIVE:
    pooled_in_set = POOLED_RANKED["ΔAIC"].le(
        DELTA_AIC_CUTOFF
    )
else:
    pooled_in_set = POOLED_RANKED["ΔAIC"].lt(
        DELTA_AIC_CUTOFF
    )
POOLED_RANKED["Retained for bootstrap"] = (
    pooled_in_set
    | POOLED_RANKED["AIC rank"].eq(1)
)
POOLED_RANKED["Fit role"] = np.where(
    POOLED_RANKED["AIC rank"].eq(1),
    "Best",
    np.where(
        POOLED_RANKED["Retained for bootstrap"],
        "Competing",
        "Outside pooled evidence set",
    ),
)
POOLED_RANKED = POOLED_RANKED.sort_values(
    ["Pooled group", "AIC", "Distribution"],
    ignore_index=True,
)
POOLED_EVIDENCE_SET = POOLED_RANKED.loc[
    POOLED_RANKED["Retained for bootstrap"]
].copy()

if POOLED_MODEL_LIMIT > 0:
    pooled_to_bootstrap = POOLED_EVIDENCE_SET.head(
        POOLED_MODEL_LIMIT
    )
else:
    pooled_to_bootstrap = POOLED_EVIDENCE_SET

pooled_gof_rows = []

for model_number, row in enumerate(
    pooled_to_bootstrap.to_dict("records"),
    start=1,
):
    item = pooled_fits[
        (row["Pooled group"], row["Distribution"])
    ]
    print(
        f"[Pooled bootstrap {model_number}/"
        f"{len(pooled_to_bootstrap)}] "
        f"{item['group']} | {item['distribution']}",
        end="",
    )
    model_start = time.time()

    gof = bootstrap_conditional_gof(
        item["key"],
        item["x"],
        item["Z"],
        [],
        item["fit"],
        n_boot=N_BOOT_POOLED,
        seed=stable_seed(
            item["group"],
            item["distribution"],
            "pooled-bootstrap",
        ),
    )
    ks_accepted = (
        bool(gof["KS p"] >= ALPHA_GOF)
        if np.isfinite(gof["KS p"])
        else pd.NA
    )
    ad_accepted = (
        bool(gof["AD p"] >= ALPHA_GOF)
        if np.isfinite(gof["AD p"])
        else pd.NA
    )
    accepted_both = (
        bool(ks_accepted and ad_accepted)
        if ks_accepted is not pd.NA
        and ad_accepted is not pd.NA
        else pd.NA
    )

    pooled_gof_rows.append(
        {
            **row,
            "KS D": gof["KS D"],
            "KS bootstrap p": gof["KS p"],
            "KS accepted": ks_accepted,
            "AD A²": gof["AD A²"],
            "AD bootstrap p": gof["AD p"],
            "AD accepted": ad_accepted,
            "Accepted by both tests": accepted_both,
            "Bootstrap valid": gof["Bootstrap valid"],
            "Bootstrap requested": gof["Bootstrap requested"],
            "Bootstrap status": gof["Bootstrap status"],
        }
    )

    print(
        f" | valid={gof['Bootstrap valid']}/"
        f"{gof['Bootstrap requested']}"
        f" | {time.time() - model_start:.1f}s"
    )

POOLED_FINAL_GOF = pd.DataFrame(pooled_gof_rows)


# Optional audit against the earlier T1 pooled-distribution workbook.
if POOLED_REFERENCE_PATH is not None:
    pooled_reference = pd.read_excel(
        POOLED_REFERENCE_PATH,
        sheet_name="S3_Fits_all",
    )
    pooled_reference = pooled_reference.loc[
        pooled_reference["group"].isin(["All", "PR", "BTW"])
        & pooled_reference["dist"].isin(
            list(family_inventory)
        )
    ][
        [
            "group",
            "dist",
            "AIC",
            "KS_p_boot",
            "AD_p_boot",
            "accepted",
        ]
    ].rename(
        columns={
            "group": "Pooled group",
            "dist": "Distribution key",
            "AIC": "Reference T1 AIC",
            "KS_p_boot": "Reference T1 KS p",
            "AD_p_boot": "Reference T1 AD p",
            "accepted": "Reference T1 accepted",
        }
    )
    POOLED_RANKED = POOLED_RANKED.merge(
        pooled_reference,
        on=["Pooled group", "Distribution key"],
        how="left",
        validate="one_to_one",
    )
    POOLED_RANKED["T1 AIC audit difference"] = (
        POOLED_RANKED["AIC"]
        - POOLED_RANKED["Reference T1 AIC"]
    )
else:
    POOLED_RANKED["Reference T1 AIC"] = np.nan
    POOLED_RANKED["Reference T1 KS p"] = np.nan
    POOLED_RANKED["Reference T1 AD p"] = np.nan
    POOLED_RANKED["Reference T1 accepted"] = pd.NA
    POOLED_RANKED["T1 AIC audit difference"] = np.nan

display(
    POOLED_FINAL_GOF[
        [
            "Pooled group",
            "Distribution",
            "Fit role",
            "AIC",
            "ΔAIC",
            "KS bootstrap p",
            "AD bootstrap p",
            "Accepted by both tests",
        ]
    ].round(4)
)


[Pooled bootstrap 1/6] All | Generalized gamma | valid=499/499 | 7.6s
[Pooled bootstrap 2/6] BTW | Gamma | valid=499/499 | 0.1s
[Pooled bootstrap 3/6] BTW | Pearson type III | valid=499/499 | 4.4s
[Pooled bootstrap 4/6] BTW | Generalized gamma | valid=499/499 | 8.6s
[Pooled bootstrap 5/6] PR | Generalized gamma | valid=499/499 | 7.1s
[Pooled bootstrap 6/6] PR | Weibull | valid=499/499 | 1.2s


,Pooled group,Distribution,Fit role,AIC,ΔAIC,KS bootstrap p,AD bootstrap p,Accepted by both tests
0,All,Generalized gamma,Best,2452.0851,0.0000,0.002,0.002,False
1,BTW,Gamma,Best,1663.3318,0.0000,0.170,0.062,True
2,BTW,Pearson type III,Competing,1664.0582,0.7264,0.076,0.016,False
3,BTW,Generalized gamma,Competing,1664.6508,1.3190,0.028,0.028,False
4,PR,Generalized gamma,Best,651.4628,0.0000,0.014,0.022,False
5,PR,Weibull,Competing,652.0063,0.5435,0.016,0.016,False


In [9]:
# ============================================================
# 9. Final model decision and pair-versus-pooled comparison
# ============================================================

def bool_or_false(value):
    return bool(value) if not pd.isna(value) else False


final_decision_rows = []

for pair in AFT_EVIDENCE_SET["Pair"].drop_duplicates():
    evidence = AFT_FINAL_GOF.loc[
        AFT_FINAL_GOF["Pair"].eq(pair)
    ].sort_values("AIC")

    adequate = evidence.loc[
        evidence["Accepted by both tests"].apply(
            bool_or_false
        )
        & evidence["Positive-support requirement met"].apply(
            bool_or_false
        )
        & evidence["Bootstrap status"].eq("OK")
    ]

    if adequate.empty:
        best_evidence = evidence.iloc[0]
        final_decision_rows.append(
            {
                "Pair": pair,
                "Final decision": (
                    "No adequate model in the ΔAIC evidence set"
                ),
                "Selected distribution": pd.NA,
                "Selected covariates": pd.NA,
                "AIC": pd.NA,
                "ΔAIC": pd.NA,
                "KS bootstrap p": pd.NA,
                "AD bootstrap p": pd.NA,
                "Lowest-AIC tested distribution": (
                    best_evidence["Distribution"]
                ),
                "Lowest-AIC test outcome": (
                    "Accepted"
                    if bool_or_false(
                        best_evidence[
                            "Accepted by both tests"
                        ]
                    )
                    else "Rejected / unavailable"
                ),
            }
        )
    else:
        selected = adequate.iloc[0]
        final_decision_rows.append(
            {
                "Pair": pair,
                "Final decision": (
                    "Selected: lowest AIC among models "
                    "accepted by both bootstrap tests"
                ),
                "Selected distribution": selected["Distribution"],
                "Selected covariates": selected[
                    "Final covariates"
                ],
                "AIC": selected["AIC"],
                "ΔAIC": selected["ΔAIC"],
                "KS bootstrap p": selected["KS bootstrap p"],
                "AD bootstrap p": selected["AD bootstrap p"],
                "Lowest-AIC tested distribution": (
                    evidence.iloc[0]["Distribution"]
                ),
                "Lowest-AIC test outcome": (
                    "Accepted"
                    if bool_or_false(
                        evidence.iloc[0][
                            "Accepted by both tests"
                        ]
                    )
                    else "Rejected / unavailable"
                ),
            }
        )

FINAL_MODEL_DECISIONS = pd.DataFrame(final_decision_rows)


# Descriptive comparison to each target class's best pooled model.
pooled_parent_best = (
    POOLED_FINAL_GOF.loc[
        POOLED_FINAL_GOF["Pooled group"].isin(["PR", "BTW"])
        & POOLED_FINAL_GOF["AIC rank"].eq(1)
    ][
        [
            "Pooled group",
            "Distribution",
            "KS D",
            "KS bootstrap p",
            "AD A²",
            "AD bootstrap p",
            "Accepted by both tests",
        ]
    ]
    .rename(
        columns={
            "Pooled group": "Target",
            "Distribution": "Pooled best distribution",
            "KS D": "Pooled KS D",
            "KS bootstrap p": "Pooled KS p",
            "AD A²": "Pooled AD A²",
            "AD bootstrap p": "Pooled AD p",
            "Accepted by both tests": (
                "Pooled accepted by both tests"
            ),
        }
    )
)

PAIR_VS_POOLED = AFT_FINAL_GOF.merge(
    pooled_parent_best,
    on="Target",
    how="left",
    validate="many_to_one",
)

def comparison_status(row):
    pair_ok = bool_or_false(row["Accepted by both tests"])
    pooled_ok = bool_or_false(
        row["Pooled accepted by both tests"]
    )
    if pair_ok and pooled_ok:
        return "Both pair-specific AFT and pooled parent accepted"
    if pair_ok and not pooled_ok:
        return (
            "Pair-specific AFT accepted; pooled parent rejected"
        )
    if not pair_ok and pooled_ok:
        return (
            "Pair-specific AFT rejected; pooled parent accepted"
        )
    return "Both rejected or a bootstrap result is unavailable"

PAIR_VS_POOLED["Descriptive comparison"] = (
    PAIR_VS_POOLED.apply(comparison_status, axis=1)
)
PAIR_VS_POOLED["Comparison caution"] = (
    "Different sample scopes: do not subtract p-values or "
    "directly compare pooled and pair-specific AIC values"
)

display(FINAL_MODEL_DECISIONS.round(4))


,Pair,Final decision,Selected distribution,Selected covariates,AIC,ΔAIC,KS bootstrap p,AD bootstrap p,Lowest-AIC tested distribution,Lowest-AIC test outcome
0,BTW_following_4W,Selected: lowest AIC among models accepted by ...,Generalized gamma,"Speed_Difference, Target_Speed_km/hr, Off_cent...",564.886204,0.0,0.734,0.47,Generalized gamma,Accepted
1,BTW_following_MT_2W,No adequate model in the ΔAIC evidence set,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Gamma,Rejected / unavailable
2,BTW_following_MT_3W,Selected: lowest AIC among models accepted by ...,Inverse Gaussian,"Target_Speed_km/hr, Speed_Difference",381.460495,0.0,0.134,0.142,Inverse Gaussian,Accepted
3,BTW_following_NMT_2W,Selected: lowest AIC among models accepted by ...,Inverse Gaussian,Speed_Difference,86.484066,0.0,0.368,0.382,Inverse Gaussian,Accepted
4,BTW_following_NMT_3W,Selected: lowest AIC among models accepted by ...,Gamma,Target_Speed_km/hr,293.911426,0.0,0.634,0.328,Gamma,Accepted
5,PR_following_4W,Selected: lowest AIC among models accepted by ...,Generalized gamma,(none),130.432732,0.011024,0.196,0.22,Weibull,Rejected / unavailable
6,PR_following_MT_3W,Selected: lowest AIC among models accepted by ...,Weibull,Target_Speed_km/hr,262.567836,0.0,0.872,0.984,Weibull,Accepted
7,PR_following_NMT_3W,No adequate model in the ΔAIC evidence set,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Generalized gamma,Rejected / unavailable


In [10]:
# ============================================================
# 10. Method record and formatted Excel export
# ============================================================

METHOD_NOTES = pd.DataFrame(
    {
        "Item": [
            "Final AFT inventory",
            "Within-pair ranking criterion",
            "Final evidence-set rule",
            "Akaike weights",
            "AFT conditional CDF",
            "Bootstrap null",
            "Bootstrap refitting",
            "Bootstrap p-value",
            "Bootstrap replicates",
            "GOF acceptance rule",
            "Cox–Snell residual",
            "Cox–Snell event status",
            "Cox–Snell plot",
            "Cox–Snell role",
            "Pooled benchmark",
            "Pooled comparison restriction",
            "Optimization actually used",
            "AICc sensitivity column",
            "Selection-adjustment scope",
            "Observed headway window caution",
        ],
        "Specification": [
            "41 final pair–distribution AFT models across 8 pairs",
            "Final AIC compared only among models fitted to the same pair",
            (
                "Minimum-AIC model plus competitors with "
                + (
                    f"ΔAIC ≤ {DELTA_AIC_CUTOFF:g}"
                    if CUTOFF_IS_INCLUSIVE
                    else f"ΔAIC < {DELTA_AIC_CUTOFF:g}"
                )
            ),
            "Computed across all final candidate models within each pair",
            "F(t|z) = F0(t exp(−zᵀβ))",
            (
                "Conditional PIT values tested against Uniform(0,1) "
                "with KS and Anderson–Darling statistics"
            ),
            (
                "Observed covariates held fixed; complete same-family "
                "AFT model refitted in every parametric-bootstrap replicate"
            ),
            (
                "p = (1 + number of bootstrap statistics at least as "
                "large as observed)/(1 + number of valid replicates)"
            ),
            (
                f"AFT={N_BOOT_FINAL}; pooled={N_BOOT_POOLED}; "
                f"minimum valid fraction={MIN_VALID_BOOT_FRACTION:.0%}"
            ),
            f"Accepted when both bootstrap p-values ≥ {ALPHA_GOF:g}",
            "r = −log S0(t exp(−zᵀβ))",
            (
                "All observations treated as events because no "
                "censoring indicator is present"
            ),
            (
                "Nelson–Aalen cumulative hazard of residuals versus "
                "H(r)=r with a pointwise unit-exponential simulation envelope"
            ),
            (
                "Graphical corroboration only; formal acceptance is "
                "based on the refitted bootstrap KS and AD tests"
            ),
            (
                "All, PR, and BTW pooled samples, fitted over the same "
                "six candidate families and subjected to the same evidence rule"
            ),
            (
                "Pooled versus pair-specific results are descriptive; "
                "AIC is not compared across different sample scopes and "
                "p-value magnitude is not treated as fit improvement"
            ),
            (
                "Bounded L-BFGS-B with multiple starts and Powell fallback "
                "when needed; not Nelder–Mead"
            ),
            (
                "AICc is reported for sensitivity but final ranking follows "
                "the requested and prior-pipeline AIC criterion"
            ),
            (
                "Bootstrap p-values are conditional on the already selected "
                "family and covariate set; the entire selection algorithm is "
                "not repeated inside each replicate"
            ),
            (
                "The likelihood treats observed 0.5–5 s headways as fully "
                "observed. If the bounds are sampling truncation rather than "
                "the operational definition of a following event, a "
                "truncated-distribution likelihood is required."
            ),
        ],
    }
)


def style_workbook(path):
    workbook = load_workbook(path)
    header_fill = PatternFill("solid", fgColor="1F4E78")
    header_font = Font(color="FFFFFF", bold=True)
    thin_gray = Side(style="thin", color="D9E1F2")
    border = Border(
        left=thin_gray,
        right=thin_gray,
        top=thin_gray,
        bottom=thin_gray,
    )

    for worksheet in workbook.worksheets:
        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = worksheet.dimensions
        worksheet.sheet_view.showGridLines = False
        worksheet.page_setup.orientation = "landscape"
        worksheet.page_setup.fitToWidth = 1
        worksheet.page_setup.fitToHeight = 0
        worksheet.sheet_properties.pageSetUpPr.fitToPage = True
        worksheet.print_title_rows = "1:1"

        for cell in worksheet[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(
                horizontal="center",
                vertical="center",
                wrap_text=True,
            )

        for row in worksheet.iter_rows():
            for cell in row:
                cell.border = border
                cell.alignment = Alignment(
                    vertical="top",
                    wrap_text=True,
                )

        for column_index, column_cells in enumerate(
            worksheet.columns,
            start=1,
        ):
            maximum = 0
            for cell in column_cells:
                value = "" if cell.value is None else str(cell.value)
                maximum = max(
                    maximum,
                    max((len(line) for line in value.splitlines()), default=0),
                )
            worksheet.column_dimensions[
                get_column_letter(column_index)
            ].width = min(max(maximum + 2, 10), 42)

        worksheet.row_dimensions[1].height = 36

        headers = {
            cell.value: cell.column
            for cell in worksheet[1]
        }
        integer_headers = {
            "Model order",
            "n",
            "k",
            "AIC rank",
            "Bootstrap KS exceedances",
            "Bootstrap AD exceedances",
            "Bootstrap valid",
            "Bootstrap requested",
        }
        for header, column_index in headers.items():
            if header in integer_headers:
                number_format = "0"
            elif any(
                token in str(header)
                for token in [
                    "AIC",
                    "logLik",
                    "weight",
                    "likelihood",
                    "KS D",
                    "KS p",
                    "AD A",
                    "AD p",
                    "mass",
                    "Residual",
                    "slope",
                    "RMSE",
                    "|H(r)-r|",
                ]
            ):
                number_format = "0.0000"
            else:
                continue

            for row_index in range(2, worksheet.max_row + 1):
                worksheet.cell(
                    row=row_index,
                    column=column_index,
                ).number_format = number_format

        for p_header in [
            "KS bootstrap p",
            "AD bootstrap p",
            "Pooled KS p",
            "Pooled AD p",
        ]:
            if p_header in headers and worksheet.max_row >= 2:
                letter = get_column_letter(headers[p_header])
                worksheet.conditional_formatting.add(
                    f"{letter}2:{letter}{worksheet.max_row}",
                    CellIsRule(
                        operator="lessThan",
                        formula=[str(ALPHA_GOF)],
                        fill=PatternFill(
                            "solid",
                            fgColor="F4CCCC",
                        ),
                    ),
                )
                worksheet.conditional_formatting.add(
                    f"{letter}2:{letter}{worksheet.max_row}",
                    CellIsRule(
                        operator="greaterThanOrEqual",
                        formula=[str(ALPHA_GOF)],
                        fill=PatternFill(
                            "solid",
                            fgColor="D9EAD3",
                        ),
                    ),
                )

    workbook.save(path)


with pd.ExcelWriter(
    OUTPUT_XLSX,
    engine="openpyxl",
) as writer:
    METHOD_NOTES.to_excel(
        writer,
        sheet_name="S0_Method",
        index=False,
    )
    AFT_RANKED.to_excel(
        writer,
        sheet_name="S1_AFT_all_41_ranked",
        index=False,
    )
    AFT_EVIDENCE_SET.to_excel(
        writer,
        sheet_name="S2_AFT_dAIC2_set",
        index=False,
    )
    AFT_FINAL_GOF.to_excel(
        writer,
        sheet_name="S3_AFT_final_GOF",
        index=False,
    )
    COX_SNELL_METRICS.to_excel(
        writer,
        sheet_name="S4_CoxSnell_metrics",
        index=False,
    )
    FINAL_MODEL_DECISIONS.to_excel(
        writer,
        sheet_name="S5_Final_decisions",
        index=False,
    )
    POOLED_RANKED.to_excel(
        writer,
        sheet_name="S6_Pooled_ranked",
        index=False,
    )
    POOLED_EVIDENCE_SET.to_excel(
        writer,
        sheet_name="S7_Pooled_dAIC2_set",
        index=False,
    )
    POOLED_FINAL_GOF.to_excel(
        writer,
        sheet_name="S8_Pooled_GOF",
        index=False,
    )
    PAIR_VS_POOLED.to_excel(
        writer,
        sheet_name="S9_Pair_vs_pooled",
        index=False,
    )
    AFT_AUDIT.to_excel(
        writer,
        sheet_name="S10_AFT_refit_audit",
        index=False,
    )

style_workbook(OUTPUT_XLSX)

print("Wrote:", OUTPUT_XLSX)
print("Wrote:", OUTPUT_PDF)


Wrote: Tables\T6_Final_AFT_Bootstrap_CoxSnell.xlsx
Wrote: Figures\T6_CoxSnell_Selected_AFT.pdf


In [11]:
# ============================================================
# 11. Integrity checks and compact manuscript table
# ============================================================

if len(AFT_RANKED) != 41:
    raise AssertionError("The final AFT ranking must contain 41 rows")
if AFT_EVIDENCE_SET["Pair"].nunique() != 8:
    raise AssertionError(
        "Every eligible pair must contribute at least its best model"
    )
if not AFT_AUDIT["AIC reconciled"].all():
    raise AssertionError("A final AFT refit failed the AIC audit")

if FINAL_MODEL_LIMIT == 0:
    if len(AFT_FINAL_GOF) != len(AFT_EVIDENCE_SET):
        raise AssertionError(
            "Not every AFT evidence-set model was bootstrapped"
        )
    if set(
        zip(
            AFT_FINAL_GOF["Pair"],
            AFT_FINAL_GOF["Distribution"],
        )
    ) != set(
        zip(
            AFT_EVIDENCE_SET["Pair"],
            AFT_EVIDENCE_SET["Distribution"],
        )
    ):
        raise AssertionError(
            "AFT evidence-set and bootstrap keys do not reconcile"
        )

if POOLED_MODEL_LIMIT == 0:
    if len(POOLED_FINAL_GOF) != len(POOLED_EVIDENCE_SET):
        raise AssertionError(
            "Not every pooled evidence-set model was bootstrapped"
        )

MANUSCRIPT_TABLE = AFT_FINAL_GOF[
    [
        "Pair",
        "Fit role",
        "Distribution",
        "Final covariates",
        "ΔAIC",
        "Akaike weight",
        "KS D",
        "KS bootstrap p",
        "KS accepted",
        "AD A²",
        "AD bootstrap p",
        "AD accepted",
        "Accepted by both tests",
    ]
].copy()
MANUSCRIPT_TABLE["Pair"] = (
    MANUSCRIPT_TABLE["Pair"]
    .str.replace("_following_", " → ", regex=False)
)

display(
    MANUSCRIPT_TABLE.round(
        {
            "ΔAIC": 3,
            "Akaike weight": 3,
            "KS D": 3,
            "KS bootstrap p": 3,
            "AD A²": 3,
            "AD bootstrap p": 3,
        }
    )
)

print(
    "\nFinal AFT models tested:",
    len(AFT_FINAL_GOF),
)
print(
    "Accepted by both tests:",
    sum(
        bool_or_false(value)
        for value in AFT_FINAL_GOF[
            "Accepted by both tests"
        ]
    ),
)
print(
    "Pairs with an adequate final model:",
    int(
        FINAL_MODEL_DECISIONS[
            "Selected distribution"
        ].notna().sum()
    ),
    "of",
    len(FINAL_MODEL_DECISIONS),
)


,Pair,Fit role,Distribution,Final covariates,ΔAIC,Akaike weight,KS D,KS bootstrap p,KS accepted,AD A²,AD bootstrap p,AD accepted,Accepted by both tests
0,BTW → 4W,Best,Generalized gamma,"Speed_Difference, Target_Speed_km/hr, Off_cent...",0.000,0.585,0.031,0.734,True,0.290,0.470,True,True
1,BTW → 4W,Competing,Pearson type III,"Speed_Difference, Target_Speed_km/hr, Off_cent...",0.688,0.415,0.035,0.532,True,0.278,0.553,True,True
2,BTW → MT_2W,Best,Gamma,"Speed_Difference, Target_Speed_km/hr",0.000,0.659,0.089,0.168,True,0.849,0.030,False,False
3,BTW → MT_3W,Best,Inverse Gaussian,"Target_Speed_km/hr, Speed_Difference",0.000,0.490,0.059,0.134,True,0.576,0.142,True,True
4,BTW → MT_3W,Competing,Log-normal,"Target_Speed_km/hr, Speed_Difference",0.390,0.403,0.056,0.150,True,0.589,0.124,True,True
5,BTW → NMT_2W,Best,Inverse Gaussian,Speed_Difference,0.000,0.340,0.104,0.368,True,0.414,0.382,True,True
6,BTW → NMT_2W,Competing,Log-normal,Speed_Difference,0.559,0.257,0.104,0.340,True,0.436,0.320,True,True
7,BTW → NMT_2W,Competing,Gamma,Speed_Difference,1.331,0.175,0.117,0.168,True,0.536,0.154,True,True
8,BTW → NMT_3W,Best,Gamma,Target_Speed_km/hr,0.000,0.305,0.052,0.634,True,0.441,0.328,True,True
9,BTW → NMT_3W,Competing,Pearson type III,Target_Speed_km/hr,0.478,0.240,0.056,0.526,True,0.405,0.440,True,True



Final AFT models tested: 17
Accepted by both tests: 14
Pairs with an adequate final model: 6 of 8
